In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    f1_score,
    classification_report
)
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

In [0]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

In [0]:
# Load the engineered feature table from Delta catalog
df = spark.read.table("novasend_fulfillment.processed.features_engineered").toPandas()

# Reproduce the exact same split used in modeling
X = df.drop(columns=["Late_delivery_risk"])
y = df["Late_delivery_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Stepwise selected features
SELECTED_FEATURES = [
    "Days_for_shipment_scheduled",
    "order_month",
    "order_quarter",
    "region_late_rate",
    "Type_TRANSFER",
    "Customer_Segment_Corporate",
    "Market_LATAM",
    "Market_USCA",
    "Order_Status_CLOSED",
    "Order_Status_ON_HOLD",
    "Order_Status_PENDING",
    "Order_Status_PROCESSING",
    "Order_Status_SUSPECTED_FRAUD",
    "Shipping_Mode_Same_Day",
    "discount_tier_aggressive"
]

# Reduce to selected features
X_train_tree = X_train[SELECTED_FEATURES]
X_test_tree = X_test[SELECTED_FEATURES]

print(f"Train shape: {X_train_tree.shape}")
print(f"Test shape: {X_test_tree.shape}")

In [0]:
# Retrain the tuned XGBoost model using the best params from RandomizedSearchCV
xgb_tuned = XGBClassifier(
    subsample=0.9,
    n_estimators=700,
    max_depth=6,
    learning_rate=0.05,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_tuned.fit(X_train_tree, y_train)

# Generate predictions and probability scores on the held-out test set
y_pred = xgb_tuned.predict(X_test_tree)
y_prob = xgb_tuned.predict_proba(X_test_tree)[:, 1]

# Confirm metrics match from what was previously logged in MLflow
auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)

print(f"AUC: {auc:.4f}")
print(f"F1:  {f1:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [0]:
# Compute confusion matrix on test set predictions
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix with labels matching our business context
fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["On Time", "Late"]
)

disp.plot(
    ax=ax,
    colorbar=False,
    cmap="Blues"
)

ax.set_title("Confusion Matrix — XGBoost Tuned (Test Set)", fontsize=13, pad=12)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)

plt.tight_layout()
plt.show()

# Print raw counts for reference
print(f"True Negatives  (On Time, predicted On Time): {cm[0][0]:,}")
print(f"False Positives (On Time, predicted Late):    {cm[0][1]:,}")
print(f"False Negatives (Late, predicted On Time):    {cm[1][0]:,}")
print(f"True Positives  (Late, predicted Late):       {cm[1][1]:,}")

In [0]:
# Compute ROC curve: fpr/tpr at every probability threshold
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))

# Plot the model ROC curve
ax.plot(
    fpr, tpr,
    color="steelblue",
    linewidth=2,
    label=f"XGBoost Tuned (AUC = {auc_score:.4f})"
)

# Plot the random classifier baseline AUC of 0.50
ax.plot(
    [0, 1], [0, 1],
    color="gray",
    linewidth=1,
    linestyle="--",
    label="Random Classifier (AUC = 0.50)"
)

ax.set_title("ROC Curve — XGBoost Tuned (Test Set)", fontsize=13, pad=12)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [0]:
# TreeExplainer is optimized for tree-based models like XGBoost
# Much faster than the generic KernelExplainer on large datasets
explainer = shap.TreeExplainer(xgb_tuned)

# Compute SHAP values on the test set
shap_values = explainer.shap_values(X_test_tree)

# Summary plot showing feature importance and direction of effect
# Each dot is one prediction, color shows feature value (red=high, blue=low)
shap.summary_plot(
    shap_values,
    X_test_tree,
    plot_type="dot",
    max_display=15,    # show all 15 selected features
    show=True
)

In [0]:
# Bar chart version of above plot
shap.summary_plot(
    shap_values,
    X_test_tree,
    plot_type="bar",
    max_display=15,
    show=True
)

In [0]:
# Final model summary
print("NOVASEND FULFILLMENT RISK — FINAL MODEL SUMMARY")
print("-" * 60)

print(f"""
Selected Model:      XGBoost (tuned)
Feature Set:         Stepwise selected — 15 of 34 features
Train Size:          {X_train_tree.shape[0]:,} orders
Test Size:           {X_test_tree.shape[0]:,} orders

Hyperparameters:
  n_estimators:      700
  max_depth:         6
  learning_rate:     0.05
  subsample:         0.9
  colsample_bytree:  0.8

Test Set Performance:
  AUC:               0.7841
  F1:                0.6920
  Precision (Late):  0.85
  Recall (Late):     0.58

Confusion Matrix:
  True Negatives:    14,317  (on time, predicted on time)
  False Positives:    1,991  (on time, predicted late)
  False Negatives:    8,270  (late, predicted on time)
  True Positives:    11,526  (late, predicted late)

Top Predictors (SHAP):
  1. Days_for_shipment_scheduled  — dominant signal (1.67)
  2. Type_TRANSFER                — strong negative risk signal (0.85)
  3. Order_Status_PENDING         — moderate positive risk signal (0.47)
  4. Order_Status_PROCESSING      — moderate positive risk signal (0.45)
  5. Order_Status_SUSPECTED_FRAUD — moderate negative risk signal (0.17)

MLflow Experiment:   /novasend-fulfillment-risk
Run Name:            xgboost_tuned
""")

print("=" * 60)
print("Evaluation notebook complete. Next: FastAPI deployment.")
print("=" * 60)